# Your first ADNI-style analysis

**Synthetic exercise.** Count fictional people who completed a cognitive assessment at their index visit. The lesson uses invented identifiers, scores and group assignments.

In [ ]:
import pandas as pd

index_visits = pd.DataFrame(
    [
        ("SYN-A", "baseline", "CN"), ("SYN-B", "baseline", "CN"),
        ("SYN-C", "baseline", "MCI"), ("SYN-D", "baseline", "MCI"),
        ("SYN-E", "baseline", "AD"), ("SYN-F", "baseline", "AD"),
    ],
    columns=["person", "visit", "diagnosis"],
)
assessments = pd.DataFrame(
    [
        ("SYN-A", "baseline", 27), ("SYN-A", "follow-up", 26),
        ("SYN-B", "follow-up", 25), ("SYN-C", "baseline", None),
        ("SYN-D", "baseline", 22), ("SYN-E", "baseline", 17),
    ],
    columns=["person", "visit", "score"],
)
print(f"{len(index_visits)} index rows, {index_visits['person'].nunique()} people")
print(f"{len(assessments)} assessment rows, {assessments['person'].nunique()} people")

## 1. Check the keys

A person can have several assessments. Each person-and-visit pair appears at most once in each table for this exercise.

In [ ]:
keys = ["person", "visit"]
for table in (index_visits, assessments):
    assert not table[keys].isna().any().any()
    assert not table.duplicated(keys).any()

## 2. Join onto the index visits

A left join keeps all six people, including those whose assessment is missing. The merge indicator distinguishes missing records from incomplete rows.

In [ ]:
joined = index_visits.merge(
    assessments, on=keys, how="left", validate="one_to_one", indicator=True
)
assert len(joined) == len(index_visits)

joined["status"] = "Complete"
joined.loc[joined["score"].isna(), "status"] = "Incomplete"
joined.loc[joined["_merge"].eq("left_only"), "status"] = "No match"
joined[["person", "visit", "status"]]

## 3. Count people by group

The result should have three complete assessments, one incomplete assessment, and two missing records.

In [ ]:
status_order = ["Complete", "Incomplete", "No match"]
summary = pd.crosstab(joined["diagnosis"], joined["status"])
summary = summary.reindex(index=["CN", "MCI", "AD"], columns=status_order, fill_value=0)
assert summary.to_numpy().tolist() == [[1, 0, 1], [1, 1, 0], [1, 0, 1]]
summary

## Try one change

Add an eligible baseline assessment for SYN-B. Predict which two counts change, rerun the join, and update the expected counts in the assertion.